In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

True

In [2]:
from accounts import Account

In [3]:
account = Account.get("Ed")
account

Account(name='ed', balance=10000.0, strategy='', holdings={}, transactions=[], portfolio_value_time_series=[])

In [4]:
account.buy_shares("AMZN", 3, "Because this bookstore website looks promising")

'Completed. Latest details:\n{"name": "ed", "balance": 9990.982, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 3.0060000000000002, "timestamp": "2026-06-11 11:48:37", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-06-11 11:48:37", 10029.982]], "total_portfolio_value": 10029.982, "total_profit_loss": 29.98199999999997}'

In [5]:
account.report()

'{"name": "ed", "balance": 9990.982, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 3.0060000000000002, "timestamp": "2026-06-11 11:48:37", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-06-11 11:48:37", 10029.982], ["2026-06-11 11:48:39", 10191.982]], "total_portfolio_value": 10191.982, "total_profit_loss": 191.98199999999997}'

In [6]:
account.list_transactions()

[{'symbol': 'AMZN',
  'quantity': 3,
  'price': 3.0060000000000002,
  'timestamp': '2026-06-11 11:48:37',
  'rationale': 'Because this bookstore website looks promising'}]

In [9]:
import os

os.environ["PATH"] += ":/Users/girisha/.local/bin:/usr/local/bin:/opt/homebrew/bin"

In [10]:
# Now let's use our accounts server as an MCP server

params = {"command": "uv", "args": ["run", "accounts_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()


In [20]:
mcp_tools

[Tool(name='get_balance', title=None, description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None),
 Tool(name='get_holdings', title=None, description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'integer'}, 'title': 'get_holdingsDictOutput', 'type': 'object'}, icons=None, annotations=None, meta=None),
 Tool(name='buy_shares', 

In [21]:
instructions = "You are able to manage an account for a client, and answer questions about the account."
request = "My name is Ed and my account is under the name Ed. What's my balance and my holdings?"
model = "gpt-4.1-mini"

In [22]:

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("account_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))


Ed, your current cash balance is $9,990.98 and you hold 3 shares of Amazon (AMZN) in your account. If you need any further information or assistance, feel free to ask!

### BUilding own MCP Client

In [23]:
from accounts_client import get_accounts_tools_openai, read_accounts_resource, list_accounts_tools

mcp_tools = await list_accounts_tools()
print(mcp_tools)
openai_tools = await get_accounts_tools_openai()
print(openai_tools)

[Tool(name='get_balance', title=None, description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None), Tool(name='get_holdings', title=None, description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'integer'}, 'title': 'get_holdingsDictOutput', 'type': 'object'}, icons=None, annotations=None, meta=None), Tool(name='buy_shares', ti

In [24]:
request = "My name is Ed and my account is under the name Ed. What's my balance?"

with trace("account_mcp_client"):
    agent = Agent(name="account_manager", instructions=instructions, model=model, tools=openai_tools)
    result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Ed, your current account balance is $9,990.98. Is there anything else you would like to know or do with your account?

In [25]:
context = await read_accounts_resource("ed")
print(context)

{"name": "ed", "balance": 9990.982, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 3.0060000000000002, "timestamp": "2026-06-11 11:48:37", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-06-11 11:48:37", 10029.982], ["2026-06-11 11:48:39", 10191.982], ["2026-06-11 12:43:52", 10071.982], ["2026-06-11 12:46:16", 10116.982]], "total_portfolio_value": 10116.982, "total_profit_loss": 116.98199999999997}


In [26]:
from accounts import Account
Account.get("ed").report()

'{"name": "ed", "balance": 9990.982, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 3.0060000000000002, "timestamp": "2026-06-11 11:48:37", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-06-11 11:48:37", 10029.982], ["2026-06-11 11:48:39", 10191.982], ["2026-06-11 12:43:52", 10071.982], ["2026-06-11 12:46:16", 10116.982], ["2026-06-11 12:46:18", 10212.982]], "total_portfolio_value": 10212.982, "total_profit_loss": 212.98199999999997}'